# Sprint 3 — PLN: Geração de Alertas, Classificação e Relatório Operacional

**Challenge FIAP — Forzy | Processamento de Linguagem Natural**
Solução de *digital-twin* para monitoramento e manutenção preditiva de motores elétricos industriais.

## Entregáveis cobertos por este notebook

| Etapa | Entregável | Célula |
|---|---|---|
| 1 | Sistema de geração de resumos textuais de alertas (3 templates: leve, moderado, crítico) | 5 e 6 |
| 2 | Avaliação dos textos gerados com métricas ROUGE sobre referências manuais | 7 |
| 3 | Classificação textual dos eventos (zero-shot com LLM, 5 categorias) | 8 |
| 4 | Avaliação do classificador com F1-score por categoria | 9 |
| 5 | Relatório operacional em linguagem natural com rastreabilidade ao sensor de origem | 10 |

**Modelo utilizado:** LLM via API Groq (`qwen/qwen3.8-27b`).

**Dados:** `data/dados_avaliacao.json` — conjunto de alertas-teste com resumos de referência (ground truth para o ROUGE).

> Execute as células na ordem, de cima para baixo.

## 1. Dependências

Instale as bibliotecas necessárias (basta executar uma vez por ambiente).

In [1]:
%pip install -q rouge-score scikit-learn python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração do ambiente

Carrega a chave da API a partir do arquivo `.env` na raiz do projeto (`NLP/.env`), que deve conter:

```
GROQ-API-KEY=sua_chave_aqui
```

In [2]:
import os
import json
from pathlib import Path

import requests
from dotenv import load_dotenv
from rouge_score import rouge_scorer
from sklearn.metrics import classification_report


# Resolve a raiz do projeto (pasta NLP) a partir do diretório de trabalho atual,
# funcionando tanto ao executar de notebooks/ quanto da raiz do repositório.
def resolver_base_dir():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "data" / "dados_avaliacao.json").exists():
            return candidato
    return Path.cwd()


BASE_DIR = resolver_base_dir()
DATA_PATH = BASE_DIR / "data" / "dados_avaliacao.json"

load_dotenv(BASE_DIR / ".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ-API-KEY")

print(f"Raiz do projeto: {BASE_DIR}")
print(f"Arquivo de dados: {DATA_PATH} (existe: {DATA_PATH.exists()})")
print(f"Chave da Groq carregada: {bool(GROQ_API_KEY)}")

Raiz do projeto: c:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 3\NLP
Arquivo de dados: c:\Users\Igor\Desktop\Sarak\Fiap\CP - Sprint - GS\2º Ano\Sprints\Sprint 3\NLP\data\dados_avaliacao.json (existe: True)
Chave da Groq carregada: True


## 3. Carregamento dos dados de avaliação

O arquivo `dados_avaliacao.json` traz o conjunto de alertas-teste. Cada item contém:

- `dados` — leitura bruta do sensor (motor, temperatura, aceleração, timestamp);
- `severidade_esperada` — nível do alerta (leve, moderado ou crítico);
- `resumo_referencia` — resumo escrito manualmente, usado como ground truth para o ROUGE.

In [3]:
def carregar_dados():
    try:
        with open(DATA_PATH, "r", encoding="utf-8") as f:
            dados_avaliacao = json.load(f)
        return dados_avaliacao.get("alertas_teste", [])
    except FileNotFoundError:
        print(f"Arquivo não encontrado em: {DATA_PATH}")
        return []


alertas_teste = carregar_dados()
print(f"Alertas-teste carregados: {len(alertas_teste)}")

for teste in alertas_teste:
    print(f"  - {teste['dados']['motor']} | severidade esperada: {teste['severidade_esperada']}")

Alertas-teste carregados: 3
  - MOT-PROD-001 | severidade esperada: crítico
  - MOT-UTIL-003 | severidade esperada: moderado
  - MOT-COMP-007 | severidade esperada: leve


## 4. Cliente do LLM

Função única de acesso à API da Groq, reutilizada pela geração de alertas, pela classificação
e pelo relatório operacional. Temperatura baixa (0.2) para respostas estáveis e reprodutíveis.

In [4]:
def call_llm(prompt):
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "qwen/qwen3.8-27b",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2
    }
    try:
        resp = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return str(e)

## 5. Entregável 1 — Geração de resumos textuais de alertas

Converte os parâmetros do alerta (sensor afetado, magnitude do desvio, timestamp) em frases
descritivas em português técnico acessível.

São definidos **três templates de narrativa**, com vocabulário e tom adequados a cada nível de urgência:

| Nível | Tom da narrativa |
|---|---|
| **Leve** | Informativo — indica leve desvio que deve ser monitorado, foco em observação contínua. |
| **Moderado** | Advertência — chama atenção e sugere inspeção preventiva em breve. |
| **Crítico** | Grave e diretivo — urgência, desligamento ou ação corretiva imediata. |

O prompt exige que os dados do sensor sejam repetidos na saída, garantindo a **rastreabilidade**
de cada afirmação até a leitura de origem.

In [5]:
def gerar_alerta(dados_motor, severidade):
    if severidade == "leve":
        tom = "Foque em observação contínua. Tom informativo, indicando leve desvio que deve ser monitorado."
    elif severidade == "moderado":
        tom = "Foque em atenção e recomendação preventiva. Tom de advertência, sugerindo inspeção em breve."
    else:  # crítico
        tom = "Foque em urgência, desligamento ou ação corretiva imediata. Tom grave e diretivo."

    prompt = f"""Você é um assistente do sistema de monitoramento de motores elétricos.
    Gere um alerta técnico em português baseado nos dados do sensor:
    - Motor: {dados_motor.get('motor')}
    - Temperatura: {dados_motor.get('temperatura')}°C
    - Aceleração/Vibração: {dados_motor.get('aceleracao')}g
    - Timestamp: {dados_motor.get('timestamp')}

    Diretrizes:
    {tom}
    Inclua os dados do sensor na sua resposta para rastreabilidade.
    """
    return call_llm(prompt)

## 6. Execução da geração dos alertas

Gera um alerta para cada leitura de sensor e exibe, lado a lado, o texto gerado e o resumo de referência.

In [6]:
alertas_gerados = []

print("Gerando alertas...")
for teste in alertas_teste:
    texto_alerta = gerar_alerta(teste["dados"], teste["severidade_esperada"])
    alertas_gerados.append({
        "gerado": texto_alerta,
        "referencia": teste["resumo_referencia"],
        "severidade_esperada": teste["severidade_esperada"]
    })

for i, a in enumerate(alertas_gerados):
    print(f"\n--- ALERTA {i + 1} ({a['severidade_esperada']}) ---")
    print("Gerado:", a["gerado"])
    print("Referência:", a["referencia"])

Gerando alertas...

--- ALERTA 1 (crítico) ---
Gerado: **⚠️ ALERTA CRÍTICO: INTERVENÇÃO IMEDIATA NECESSÁRIA**

**MOTOR:** MOT-PROD-001
**STATUS:** RISCO DE FALHA CATASTRÓFICA

**AÇÃO EXIGIDA:**
**DESLIGUE O MOTOR IMEDIATAMENTE!**

Os parâmetros operacionais ultrapassaram os limites de segurança aceitáveis. A combinação de temperatura elevada e vibração anômala indica desgaste severo ou desbalanceamento crítico que pode levar à queima do enrolamento ou ruptura mecânica em minutos.

**DADOS DO SENSOR (Rastreabilidade):**
*   **Timestamp:** 2026-05-19T11:48:55
*   **Temperatura:** 47°C (Acima do limite operacional seguro para esta classe de motor)
*   **Aceleração/Vibração:** 0.47g (Nível crítico de vibração)

**INSTRUÇÕES CORRETIVAS:**
1.  Inicie o procedimento de parada de emergência (E-Stop) do motor MOT-PROD-001 agora.
2.  Isole a unidade para evitar reinicialização acidental.
3.  Notifique a equipe de manutenção para inspeção física imediata antes de qualquer nova partida.
4.  Regist

## 7. Avaliação dos textos gerados — métricas ROUGE

Compara cada alerta gerado com o resumo de referência manual.

- **ROUGE-1** — sobreposição de unigramas: mede se os termos técnicos e os valores de sensor corretos aparecem.
- **ROUGE-L** — maior subsequência comum: mede se a estrutura da narrativa acompanha a referência.

Complementar ao ROUGE, o **checklist de clareza, precisão e utilidade** é aplicado sobre as saídas
abaixo e registrado no documento final unificado.

In [7]:
def avaliar_rouge(alertas_gerados):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    print("\n--- Avaliação ROUGE dos Alertas Gerados ---")
    for i, a in enumerate(alertas_gerados):
        scores = scorer.score(a["referencia"], a["gerado"])
        print(f"Alerta {i + 1} ({a['severidade_esperada']}):")
        print(f"  ROUGE-1: Precision: {scores['rouge1'].precision:.2f}, Recall: {scores['rouge1'].recall:.2f}, F1: {scores['rouge1'].fmeasure:.2f}")
        print(f"  ROUGE-L: Precision: {scores['rougeL'].precision:.2f}, Recall: {scores['rougeL'].recall:.2f}, F1: {scores['rougeL'].fmeasure:.2f}")


avaliar_rouge(alertas_gerados)


--- Avaliação ROUGE dos Alertas Gerados ---
Alerta 1 (crítico):
  ROUGE-1: Precision: 0.16, Recall: 0.93, F1: 0.27
  ROUGE-L: Precision: 0.10, Recall: 0.60, F1: 0.18
Alerta 2 (moderado):
  ROUGE-1: Precision: 0.13, Recall: 0.90, F1: 0.23
  ROUGE-L: Precision: 0.12, Recall: 0.83, F1: 0.21
Alerta 3 (leve):
  ROUGE-1: Precision: 0.16, Recall: 0.87, F1: 0.27
  ROUGE-L: Precision: 0.12, Recall: 0.65, F1: 0.20


## 8. Entregável 2 — Classificação textual dos eventos

Categorização automática **zero-shot com LLM** dos eventos registrados, nas cinco categorias exigidas:

`manutenção corretiva` · `manutenção preventiva` · `anomalia elétrica` · `anomalia mecânica` · `operação normal`

O prompt restringe a saída ao nome da categoria; um passo de normalização mapeia a resposta
para uma das classes válidas (`desconhecido` quando não há correspondência).

In [8]:
def classificar_alerta(alerta):
    categorias = ["manutenção corretiva", "manutenção preventiva", "anomalia elétrica", "anomalia mecânica", "operação normal"]
    prompt = f"""Classifique o seguinte alerta de motor elétrico estritamente em UMA das categorias abaixo.
    Retorne APENAS o nome da categoria, sem textos adicionais.
    Categorias: {', '.join(categorias)}

    Alerta: {alerta}
    """
    resposta = call_llm(prompt).lower()
    for c in categorias:
        if c in resposta:
            return c
    return "desconhecido"

In [9]:
print("Classificando alertas...")

# Gabarito: o alerta crítico exige manutenção corretiva, o moderado uma preventiva
# e o leve caracteriza operação normal sob monitoramento.
y_true = ["manutenção corretiva", "manutenção preventiva", "operação normal"]
y_pred = []

for a in alertas_gerados:
    cat = classificar_alerta(a["gerado"])
    y_pred.append(cat)
    print(f"Alerta ({a['severidade_esperada']}) classificado como: {cat}")

Classificando alertas...
Alerta (crítico) classificado como: manutenção corretiva
Alerta (moderado) classificado como: manutenção preventiva
Alerta (leve) classificado como: manutenção preventiva


## 9. Avaliação do classificador — F1-score por categoria

Relatório com precision, recall e F1-score para cada uma das categorias previstas.

In [10]:
print("\n--- Avaliação F1-Score da Classificação ---")
print(classification_report(y_true, y_pred, zero_division=0))


--- Avaliação F1-Score da Classificação ---
                       precision    recall  f1-score   support

 manutenção corretiva       1.00      1.00      1.00         1
manutenção preventiva       0.50      1.00      0.67         1
      operação normal       0.00      0.00      0.00         1

             accuracy                           0.67         3
            macro avg       0.50      0.67      0.56         3
         weighted avg       0.50      0.67      0.56         3



## 10. Entregável 3 — Relatório de estado operacional

Sumariza em linguagem natural, a partir dos alertas já gerados e classificados:

1. alertas emitidos;
2. equipamentos em risco;
3. tendências observadas;
4. recomendações preliminares.

O prompt exige explicitamente que **cada afirmação referencie o dado de sensor de origem**,
atendendo ao requisito de rastreabilidade.

In [11]:
def gerar_relatorio_operacional(alertas_classificados):
    prompt = """Com base na seguinte lista de alertas classificados, gere um relatório diário consolidado em linguagem natural sumarizando:
    1. Alertas emitidos
    2. Equipamentos em risco
    3. Tendências observadas
    4. Recomendações preliminares
    Cada afirmação deve referenciar o dado do sensor de origem (rastreabilidade).

    Alertas:
"""
    for a in alertas_classificados:
        prompt += f"- {a}\n"

    return call_llm(prompt)

In [12]:
print("Gerando relatório operacional consolidado...")

textos_para_relatorio = [f"{a['gerado']} -> Classificado como: {c}" for a, c in zip(alertas_gerados, y_pred)]
relatorio = gerar_relatorio_operacional(textos_para_relatorio)

print("\n--- RELATÓRIO OPERACIONAL CONSOLIDADO ---")
print(relatorio)

Gerando relatório operacional consolidado...

--- RELATÓRIO OPERACIONAL CONSOLIDADO ---
Aqui está o relatório diário consolidado baseado nos alertas recebidos:

**Relatório Diário de Monitoramento de Equipamentos**
**Data de Referência:** 19 de maio de 2026 (11:48:55)

**1. Alertas Emitidos**
Foram registrados três eventos de monitoramento no sistema nesta data. O evento mais grave é um **alerta crítico** para o motor **MOT-PROD-001**, classificado para manutenção corretiva imediata devido a parâmetros que ultrapassaram os limites de segurança. Adicionalmente, foram emitidos dois alertas técnicos de nível de atenção/observação para os motores **MOT-UTIL-003** e **MOT-COMP-007**, ambos classificados para manutenção preventiva, indicando desvios operacionais que requerem monitoramento próximo, mas não intervenção emergencial.

**2. Equipamentos em Risco**
O equipamento em risco imediato é o **MOT-PROD-001**. Os dados do sensor indicam uma condição de risco de falha catastrófica, com a te

## Conclusão

Este notebook entregou, sobre os dados do projeto:

- **Geração de alertas** em português técnico acessível, com três templates de narrativa distintos (leve, moderado e crítico) e rastreabilidade ao dado de sensor;
- **Avaliação ROUGE-1 e ROUGE-L** de cada alerta gerado contra resumos de referência escritos manualmente;
- **Classificação zero-shot** dos eventos nas cinco categorias operacionais, avaliada por **F1-score por categoria**;
- **Relatório operacional consolidado** em linguagem natural, com alertas emitidos, equipamentos em risco, tendências e recomendações preliminares.

O contexto operacional produzido aqui (resumos de alerta e estado do ativo) é injetado como contexto
adicional no assistente conversacional do notebook `sprint4_pln_rag.ipynb`.